# Milestone 4 - Task 6: Email Preprocessing Pipeline

**Owner:** Sanjeewa Narayana  
**Objective:** A single reproducible pipeline that turns the raw cleaned email CSV into the **email feature matrix** end-to-end: normalize text + extract body features.

### Architecture — two independent tracks
| Track | Source | Rows | Tasks |
|---|---|---|---|
| Email | phishing_email.csv | 82,078 | T1 -> T2 -> T5 -> **T6** |
| URL   | malicious_phish.csv | 641,119 | T3 (standalone) |

This pipeline builds the **email track** only. The URL matrix (T3) stays separate. M4-T7 exports both. Header features (T4) are excluded — the dataset has no raw headers (see T4).

## How phishing language is captured (design note)
The **primary** language signal is the `text_clean` column produced here: at modeling time (M5) it is vectorized with **TF-IDF / n-grams**, so the classifier learns phishing vocabulary — including synonyms, misspellings, and variants — directly from the data, with no hand-maintained word list.

`urgency_score` is kept only as a small, **interpretable auxiliary** feature. It is hardened over a naive exact-word list:
- **stem patterns** — `verif` matches verify / verified / verifying / verification
- **`\b` word boundaries** — avoids substring false positives (e.g. `limit` no longer fires on *unlimited*)
It is not the main mechanism for detecting phishing language.

## Step 1 — Configuration (notebook equivalent of CLI args)

In [1]:
from pathlib import Path
import csv, re
csv.field_size_limit(10_000_000)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
INPUT_PATH  = PROJECT_ROOT / 'data' / 'processed' / 'emails_clean.csv'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'email_features.csv'
print('Input :', INPUT_PATH)
print('Output:', OUTPUT_PATH)

Input : <PROJECT_ROOT>/data/processed/emails_clean.csv
Output: <PROJECT_ROOT>/data/processed/email_features.csv


## Step 2 — Pipeline stages
`normalize_text` produces the cleaned text for TF-IDF in M5; `body_features` computes the 5 numeric features on the RAW text (so HTML/links survive).

In [2]:
# --- Auxiliary urgency cue (NOT the primary signal; see design note) ---
# Stem patterns + word boundaries generalize over morphological variants
# and avoid substring false positives.
URGENCY_STEMS = [
    r'urgent', r'immediat', r'verif', r'suspend', r'limit', r'restrict',
    r'action required', r'click here', r'confirm', r'account', r'password',
    r'updat', r'expir', r'unusual activ', r'security alert',
    r'hurry', r'act now', r'last chance', r'time.?sensitive',
]
URGENCY_RE = re.compile(r'\b(?:' + '|'.join(URGENCY_STEMS) + r')', re.IGNORECASE)

def urgency_score(text):
    """Count urgency-cue occurrences (stem + word-boundary matching)."""
    return len(URGENCY_RE.findall(str(text)))

def normalize_text(text):
    t = str(text).lower()
    t = re.sub(r'<[^>]+>', ' ', t)            # strip HTML tags
    t = re.sub(r'http\S+|www\.\S+', ' ', t)  # strip URLs
    t = re.sub(r'\S+@\S+', ' ', t)            # strip emails
    return re.sub(r'\s+', ' ', t).strip()

def body_features(text):
    raw = str(text)
    words = raw.split()
    tags = len(re.findall(r'<[^>]+>', raw))
    return {
        'urgency_score':   urgency_score(raw),
        'link_count':      len(re.findall(r'https?://[^\s]+', raw)),
        'html_ratio':      round(tags / len(raw), 4) if raw else 0,
        'word_count':      len(words),
        'avg_word_length': round(sum(len(w) for w in words)/len(words), 2) if words else 0,
    }

## Step 3 — Run the pipeline

In [3]:
OUT_COLS = ['text_clean','label','urgency_score','link_count',
            'html_ratio','word_count','avg_word_length']

with INPUT_PATH.open('r', encoding='utf-8', newline='') as f:
    rows = list(csv.DictReader(f))

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open('w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=OUT_COLS); w.writeheader()
    for r in rows:
        raw = r['text_combined']
        rec = {'text_clean': normalize_text(raw), 'label': r['label']}
        rec.update(body_features(raw))
        w.writerow(rec)

print(f'Pipeline complete. {len(rows)} rows -> {OUTPUT_PATH}')
print(f'Columns ({len(OUT_COLS)}): {OUT_COLS}')

Pipeline complete. 82078 rows -> <PROJECT_ROOT>/data/processed/email_features.csv
Columns (7): ['text_clean', 'label', 'urgency_score', 'link_count', 'html_ratio', 'word_count', 'avg_word_length']


## Step 4 — Verify output

In [4]:
with OUTPUT_PATH.open('r', encoding='utf-8') as f:
    head = [next(f) for _ in range(3)]
print(''.join(head))

text_clean,label,urgency_score,link_count,html_ratio,word_count,avg_word_length
hpl nom may 25 2001 see attached file hplno 525 xls hplno 525 xls,0,0,0,0.0,14,3.71
nom actual vols 24 th forwarded sabrae zajac hou ect 05 30 2001 12 07 pm enron capital trade resources corp eileen ponton 05 29 2001 08 37 davilal txu com cstonel txu com mjones 7 txu com hpl scheduling enron com liz bellamy enron com szajac enron com cc subject nom actual vols 24 th agree nomination 33 750 forwarded eileen ponton houston pefs pec 05 29 01 08 36 charlie stone eileen ponton melissa jones com hpl scheduling enron com liz bellamy enron com szajac enron com 05 25 01 subject nom actual vols 24 th 04 23 pm agree nominated volume records reflect following nom schedule 30 rate eff 0900 hrs hour beginning 1400 hrs 6 250 60 rate eff 1400 hrs hour beginning 1700 hrs 7 500 30 rate eff 1700 hrs hour beginning 0900 hrs 20 000 total nominated 33 750 please review source data let us know agree thanks ccs eileen ponton 05 25

## Step 5 — Upload to S3 (run in terminal; do NOT commit the CSV)
```bash
aws s3 cp data/processed/email_features.csv \
  s3://email-security-pipeline-datasets/datasets/processed/email_features.csv \
  --profile lab-user
```

Hand off both `email_features.csv` (this) and `url_features.csv` (T3) to M4-T7.